# Stage 3: Sample Uniqueness and Weights
This notebook implements concurrent-label counting, uniqueness, sequential bootstrap, and time decay weights.


In [ ]:
# ============================================================
# TICKER PARAMETER — change this to run for any stock
# ============================================================
TICKER = 'NVDA'   # Options: AAPL AMZN BAC GOOGL JNJ JPM MSFT NVDA UNH XOM

import os, sys
sys.path.insert(0, os.path.abspath('../..'))

DATA_RAW     = f'../../data/raw/{TICKER}_raw.csv'
DATA_CLEAN   = f'../../data/processed/per_stock/{TICKER}_clean.parquet'
DATA_CUSUM   = f'../../data/processed/per_stock/{TICKER}_cusum_events.parquet'
DATA_BARS    = f'../../data/processed/per_stock/{TICKER}_dollar_bars.parquet'
DATA_LABELS  = f'../../data/processed/per_stock/{TICKER}_labels.parquet'
DATA_WEIGHTS = f'../../data/processed/per_stock/{TICKER}_weights.parquet'
DATA_FRACDIFF= f'../../data/processed/per_stock/{TICKER}_fracdiff.parquet'
DATA_FEATURES= f'../../data/processed/per_stock/{TICKER}_ts_features.parquet'
DATA_BSADF   = f'../../data/processed/per_stock/{TICKER}_bsadf.parquet'
FIG_DIR      = f'../../reports/figures/per_stock/{TICKER}'
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs('../../data/processed/per_stock', exist_ok=True)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sys
import os

sys.path.insert(0, os.path.abspath('../..'))
from src.sample_weights import num_co_events, sample_tw, get_ind_matrix, seq_bootstrap, get_return_attribution, get_time_decay, get_sample_weight

plt.style.use('seaborn-v0_8-darkgrid')


## 1. Load Data


In [ ]:
close_df = pd.read_parquet(f'../../data/processed/per_stock/{TICKER}_clean.parquet')
close = close_df['Adj Close']

labels = pd.read_parquet(f'../../data/processed/per_stock/{TICKER}_labels.parquet')
labels.head()


## 2. Concurrency Count


In [ ]:
co_events = num_co_events(close.index, labels['t1'], labels.index)
print("Concurrency counts computed.")


## 3. Average Uniqueness


In [ ]:
avg_uniqueness = sample_tw(labels['t1'], co_events, labels.index)
avg_uniqueness.describe()


## 4. Sequential vs Standard Bootstrap


In [ ]:
ind_matrix = get_ind_matrix(close.index, labels['t1'])

# Standard bootstrap (100 draws)
np.random.seed(42)
std_bootstrap = np.random.choice(range(ind_matrix.shape[1]), size=100, replace=True)

# Sequential bootstrap (100 draws)
seq_boot = seq_bootstrap(ind_matrix, s_length=100)

# Evaluate average uniqueness of drawn samples
std_u = avg_uniqueness.iloc[std_bootstrap].mean()
seq_u = avg_uniqueness.iloc[seq_boot].mean()

print(f"Standard Bootstrap Average Uniqueness: {std_u:.4f}")
print(f"Sequential Bootstrap Average Uniqueness: {seq_u:.4f}")


## 5. Return-Attribution Weights


In [ ]:
ret_weights = get_return_attribution(labels)
ret_weights.describe()


## 6. Time Decay


In [ ]:
decay_weights = get_time_decay(avg_uniqueness, c_lf=0.5)
decay_weights.describe()


## 7. Final Combined Weights


In [ ]:
final_weights = get_sample_weight(labels, close)
labels['weight'] = final_weights
print("Final Weights Statistics:")
display(labels['weight'].describe())


## 8. Plots


In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 15))

# a. Concurrency over time
axes[0].plot(co_events.index, co_events.values, color='teal', alpha=0.8)
axes[0].set_title('Number of Concurrent Events Over Time')
axes[0].set_ylabel('Concurrency (c_t)')

# b. Uniqueness histogram
axes[1].hist(avg_uniqueness, bins=50, color='darkorange', alpha=0.7)
axes[1].set_title('Histogram of Average Uniqueness (ū_i)')
axes[1].set_xlabel('Uniqueness')

# c. Weight distribution boxplot
axes[2].boxplot(final_weights.dropna())
axes[2].set_title('Boxplot of Final Combined Weights')
axes[2].set_ylabel('Weight')

plt.tight_layout()
plt.show()


## 9. Save Sample Weights


In [ ]:
labels[['weight']].to_parquet(f'../../data/processed/per_stock/{TICKER}_weights.parquet')
print(f"Saved {TICKER} data to {DATA_CLEAN}")
